# user

In [ ]:
VERSION = "0.0.8"
ASSOC = False
PIXEL_SCALE = 0.1

# lib

In [ ]:
from pathlib import Path
import numpy
from pandas import read_csv, concat
from tqdm import tqdm
from plots import (
    image_fitting,
    histos,
    ellips_model,
    residuals,
    case_result_param,
    vignettes,
    case_result,
    fmf_values,
    DIRS,
    comparaison,
    logged_values,
    x_n_seric,
    get_match,
)


DATASET_DIR_ORIGIN = DIRS["pisco_euclid"] / VERSION
DATASET_DIR = DIRS["pisco_euclid_se"] / VERSION
RESULTS_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR") / f"{VERSION}++"
if ASSOC:
    RESULTS_DIR = RESULTS_DIR.parent / f"{RESULTS_DIR.name}_assoc"
CATALOG_DIR = RESULTS_DIR / "CATALOG"
IMAGES_DIR = RESULTS_DIR / "IMAGES"
PLOTS_DIR = RESULTS_DIR / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# inference

In [ ]:
# load parameters
true_avg = read_csv(DATASET_DIR_ORIGIN / "images_parameters_true.csv", sep=",", skiprows=0)
true = read_csv(DATASET_DIR_ORIGIN / "galaxies_parameters_true.csv", sep=",", skiprows=0)
PRED_FILES = sorted(list((CATALOG_DIR).glob("*_simu.csv")))
TRUE_FILES = sorted(list((DATASET_DIR).glob("*_true_galaxies_params.csv")))

# inference
pred_avg_e1, pred_avg_e2, detections, pred, slice_pred, slice_truth = [[] for _ in range(6)]
for i, (res_file, true_file) in enumerate(tqdm(zip(PRED_FILES, TRUE_FILES), total=len(PRED_FILES))):
    # load results
    df_pred = read_csv(res_file).sort_values("x_model")
    df_pred = df_pred.dropna(subset=["e2_model_err", "e1_model_err"])
    detections.append(df_pred.shape[0])
    df_true = read_csv(true_file).sort_values("x")

    # slices
    temp_pred, temp_truth = get_match(df_pred, df_true, radius=2)
    temp_pred["image"] = i
    temp_truth["image"] = i
    slice_pred.append(temp_pred)
    slice_truth.append(temp_truth)

    # predictions galaxies
    df_pred["image"] = i
    pred.append(df_pred)

    # predictions images
    avg_e1_temp, avg_e2_temp = ellips_model(
        df_pred["e1_model_err"].to_numpy(),
        df_pred["e2_model_err"].to_numpy(),
        df_pred["e1_model"].to_numpy(),
        df_pred["e2_model"].to_numpy(),
    )
    pred_avg_e1.append(avg_e1_temp)
    pred_avg_e2.append(avg_e2_temp)

# final results
pred = concat(pred, ignore_index=True)
slice_pred = concat(slice_pred, ignore_index=True)
slice_truth = concat(slice_truth, ignore_index=True)
detections = numpy.array(detections)
pred_avg_e1 = numpy.array(pred_avg_e1)
pred_avg_e2 = numpy.array(pred_avg_e2)
column_pred = ["e1_model", "e2_model", "image"]
slice_pred_avg_e = slice_pred[column_pred].groupby("image")[column_pred].mean()
column_true = ["obs_e1", "obs_e2", "image"]
slice_true_avg_e = slice_truth[column_true].groupby("image")[column_true].mean()

# post process

In [ ]:
top = 0.05
mask = (true_avg["avg_g1"].abs() <= top) & (true_avg["avg_g2"].abs() <= top)
images_to_keep = mask[mask].index.to_numpy()

pred_avg_e1 = pred_avg_e1[mask]
pred_avg_e2 = pred_avg_e2[mask]
true_avg = true_avg[mask]
detections = detections[mask]

slice_pred = slice_pred[slice_pred["image"].isin(images_to_keep)]
slice_truth = slice_truth[slice_truth["image"].isin(images_to_keep)]
slice_pred_avg_e = slice_pred_avg_e[slice_pred_avg_e["image"].isin(images_to_keep)]
slice_true_avg_e = slice_true_avg_e[slice_true_avg_e["image"].isin(images_to_keep)]
pred = pred[pred["image"].isin(images_to_keep)]
true = true[true["image"].isin(images_to_keep)]

print(NB_GAL_TOTAL := true.shape[0])
print(NB_GAL_PRED := pred.shape[0])
print(NB_GOOD_GAL := slice_pred.shape[0])

del df_pred, df_true, avg_e1_temp, avg_e2_temp, PRED_FILES, TRUE_FILES, i, column_pred
del res_file, true_file, mask, top, images_to_keep, temp_pred, temp_truth, column_true

# histo

In [ ]:
bins = 100
log = True
fmf_values(bins, pred, PLOTS_DIR, log=log)
del log, bins

In [ ]:
bins = 100
logged_values(bins, pred, true, PIXEL_SCALE, PLOTS_DIR)

In [ ]:
bins = 50
density = True

histos(
    detections,
    pred,
    true,
    pred_avg_e1,
    pred_avg_e2,
    true_avg,
    PLOTS_DIR,
    PIXEL_SCALE,
    bins=bins,
    density=density,
)

del bins, density

# results

In [ ]:
case_result(
    pred_avg_e1,
    pred_avg_e2,
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GAL_PRED,
    PLOTS_DIR / "results_se.pdf",
)

In [ ]:
case_result(
    true_avg["avg_obs_e1"],
    true_avg["avg_obs_e2"],
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GAL_TOTAL,
    PLOTS_DIR / "results_se_best.pdf",
)

# results good galaxies

In [ ]:
case_result(
    slice_pred_avg_e["e1_model"].to_numpy(),
    slice_pred_avg_e["e2_model"].to_numpy(),
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GOOD_GAL,
    PLOTS_DIR / "results_good_gal_se.pdf",
)

In [ ]:
case_result(
    slice_true_avg_e["obs_e1"].to_numpy(),
    slice_true_avg_e["obs_e2"].to_numpy(),
    true_avg["avg_g1"],
    true_avg["avg_g2"],
    NB_GOOD_GAL,
    PLOTS_DIR / "results_good_gal_se_best.pdf",
)

# results by images

In [ ]:
CHOICE = 1
# CHOICE = 6578

# style_pred = slice_pred.copy()
# style_true = slice_truth.copy()
style_pred = pred.copy()
style_true = true.copy()

CHOICE_IMAGE = int(numpy.unique(slice_pred["image"])[CHOICE])
example_preds = style_pred[style_pred["image"] == CHOICE_IMAGE].reset_index(drop=True)
example_true = style_true[style_true["image"] == CHOICE_IMAGE].reset_index(drop=True)
comparaison(
    DATASET_DIR / f"{CHOICE_IMAGE:06d}_simu",
    example_preds[
        [
            "fmf_reduced_chi_2",
            "fmf_stop_reason",
            "fmf_flags",
            "x_model",
            "y_model",
            "e1_model",
            "e2_model",
        ]
    ],
    example_true[["obs_e1", "obs_e2", "x", "y"]].sort_values("x"),
    detections[CHOICE],
    true_avg["avg_g1"][CHOICE_IMAGE],
    true_avg["avg_g2"][CHOICE_IMAGE],
    PLOTS_DIR,
)

In [ ]:
if CHOICE <= 100:
    image_fitting(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)
    image_fitting(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR, transformed=True)
    residuals(DATASET_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)
    vignettes(CATALOG_DIR, IMAGES_DIR, f"{CHOICE_IMAGE:06d}_simu", PLOTS_DIR)

# results by params

In [ ]:
case_result_param(PLOTS_DIR, slice_pred["x_model"], slice_truth["x"], err=0, symbol="x")
case_result_param(PLOTS_DIR, slice_pred["y_model"], slice_truth["y"], err=0, symbol="y")

In [ ]:
case_result_param(PLOTS_DIR, slice_pred["e1_model"], slice_truth["obs_e1"], err=0, symbol="obs_e1")
case_result_param(PLOTS_DIR, slice_pred["e2_model"], slice_truth["obs_e2"], err=0, symbol="obs_e2")

In [ ]:
case_result_param(
    PLOTS_DIR, slice_pred["sersic_model"], slice_truth["sersic"], err=0, symbol="sersic"
)
case_result_param(PLOTS_DIR, slice_pred["mag_model"], slice_truth["mag"], err=0, symbol="mag")
case_result_param(
    PLOTS_DIR, slice_pred["flux_radius"] * PIXEL_SCALE, slice_truth["hlr"], err=0, symbol="hlr"
)

# sersic

In [ ]:
MIN_SERSIC = 0.2
MAX_SERSIC = 6.5
MU_X = -2.6
SIGMA_X = 1.1

x_n_seric(MIN_SERSIC, MAX_SERSIC, MU_X, SIGMA_X, PLOTS_DIR)